### exercise 1 dataset autopsy

objective initial inspection and null detection

In [1]:
import pandas as pd

url = "https://raw.githubusercontent.com/datasciencedojo/datasets/master/titanic.csv"
df = pd.read_csv(url)

# show the first 5 rows
display(df.head())

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


In [2]:
# check for null values
print('null counts per column:')
display(df.isnull().sum())

null counts per column:


,0
PassengerId,0
Survived,0
Pclass,0
Name,0
Sex,0
Age,177
SibSp,0
Parch,0
Ticket,0
Fare,0


In [3]:
# check data types
print('data types of columns:')
display(df.info())

data types of columns:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 891 entries, 0 to 890
Data columns (total 12 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   PassengerId  891 non-null    int64  
 1   Survived     891 non-null    int64  
 2   Pclass       891 non-null    int64  
 3   Name         891 non-null    object 
 4   Sex          891 non-null    object 
 5   Age          714 non-null    float64
 6   SibSp        891 non-null    int64  
 7   Parch        891 non-null    int64  
 8   Ticket       891 non-null    object 
 9   Fare         891 non-null    float64
 10  Cabin        204 non-null    object 
 11  Embarked     889 non-null    object 
dtypes: float64(2), int64(5), object(5)
memory usage: 83.7+ KB


None

### exercise 2 surgical cleaning

objective handle duplicates impute nulls and manage outliers

In [4]:
# check for and remove duplicates
duplicates_count = df.duplicated().sum()
print(f"number of duplicate rows before dropping {duplicates_count}")
df.drop_duplicates(inplace=True)
print(f"number of duplicate rows after dropping {df.duplicated().sum()}")

number of duplicate rows before dropping 0
number of duplicate rows after dropping 0


In [5]:
# impute 'embarked' with mode
most_frequent_embarked = df['Embarked'].mode()[0]
df['Embarked'] = df['Embarked'].fillna(most_frequent_embarked)
print(f"'embarked' nulls after imputation {df['Embarked'].isnull().sum()}")

'embarked' nulls after imputation 0


In [6]:
# impute 'age' with median
median_age = df['Age'].median()
df['Age'] = df['Age'].fillna(median_age)
print(f"'age' nulls after imputation {df['Age'].isnull().sum()}")

'age' nulls after imputation 0


In [7]:
# impute 'cabin' with a placeholder 'u' for unknown
df['Cabin'] = df['Cabin'].fillna('U')
print(f"'cabin' nulls after imputation {df['Cabin'].isnull().sum()}")

'cabin' nulls after imputation 0


In [8]:
# handle outliers for 'fare' using iqr method
q1_fare = df['Fare'].quantile(0.25)
q3_fare = df['Fare'].quantile(0.75)
iqr_fare = q3_fare - q1_fare
upper_bound_fare = q3_fare + 1.5 * iqr_fare
lower_bound_fare = q1_fare - 1.5 * iqr_fare
df['Fare'] = df['Fare'].clip(lower=lower_bound_fare, upper=upper_bound_fare)
print("fare outliers capped using iqr")

fare outliers capped using iqr


In [9]:
# handle outliers for 'age' using iqr method
q1_age = df['Age'].quantile(0.25)
q3_age = df['Age'].quantile(0.75)
iqr_age = q3_age - q1_age
upper_bound_age = q3_age + 1.5 * iqr_age
lower_bound_age = q1_age - 1.5 * iqr_age
df['Age'] = df['Age'].clip(lower=lower_bound_age, upper=upper_bound_age)
print("age outliers capped using iqr")

age outliers capped using iqr


In [10]:
# recheck null values after cleaning
print('null counts per column after cleaning:')
display(df.isnull().sum())

null counts per column after cleaning:


,0
PassengerId,0
Survived,0
Pclass,0
Name,0
Sex,0
Age,0
SibSp,0
Parch,0
Ticket,0
Fare,0


### exercise 3 the art of representation

objective mathematical transformations and categorical encoding

In [11]:
# create a new feature 'familysize'
df_features = df.copy()

df_features['FamilySize'] = df_features['SibSp'] + df_features['Parch'] + 1

print('new feature familysize created')

display(df_features[['SibSp', 'Parch', 'FamilySize']].head())

new feature familysize created


,SibSp,Parch,FamilySize
0,1,0,2
1,1,0,2
2,0,0,1
3,1,0,2
4,0,0,1


In [12]:
# encode 'sex' column using one-hot encoding
df_features = pd.get_dummies(
    df_features,
    columns=['Sex'],
    drop_first=True,
    prefix='Sex',
    dtype=int
)

display(df_features[['Sex_male']].head())

,Sex_male
0,1
1,0
2,0
3,0
4,1


In [13]:
df_features = pd.get_dummies(
    df_features,
    columns=['Embarked'],
    drop_first=True,
    prefix='Embarked',
    dtype=int
)

display(df_features[['Embarked_Q', 'Embarked_S']].head())

,Embarked_Q,Embarked_S
0,0,1
1,0,0
2,0,1
3,0,1
4,0,1


In [14]:
# display first few rows with new features
print('dataframe after transformations and encoding:')
display(df_features.head())

dataframe after transformations and encoding:


,PassengerId,Survived,Pclass,Name,Age,SibSp,Parch,Ticket,Fare,Cabin,FamilySize,Sex_male,Embarked_Q,Embarked_S
0,1,0,3,"Braund, Mr. Owen Harris",22.0,1,0,A/5 21171,7.2500,U,2,1,0,1
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",38.0,1,0,PC 17599,65.6344,C85,2,0,0,0
2,3,1,3,"Heikkinen, Miss. Laina",26.0,0,0,STON/O2. 3101282,7.9250,U,1,0,0,1
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",35.0,1,0,113803,53.1000,C123,2,0,0,1
4,5,0,3,"Allen, Mr. William Henry",35.0,0,0,373450,8.0500,U,1,1,0,1


### exercise 4 the integrated pipeline (leakage prevention)

objective avoid information leakage by separating scaling and imputation

In [15]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

# drop irrelevant columns for modeling
df_processed = df.drop(['PassengerId', 'Name', 'Ticket', 'Cabin'], axis=1)

# define features (x) and target (y)
x = df_processed.drop('Survived', axis=1)
y = df_processed['Survived']

# split data into training and testing sets to prevent leakage
x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.2, random_state=42)

print('data split into training and testing sets')
print(f'x_train shape {x_train.shape}')
print(f'x_test shape {x_test.shape}')

data split into training and testing sets
x_train shape (712, 7)
x_test shape (179, 7)


In [16]:
# identify numerical columns for scaling and potential imputation
# all remaining columns are numerical after previous steps except target
numerical_cols = x_train.select_dtypes(include=['int64', 'float64']).columns.tolist()

# create a preprocessing pipeline for numerical features
# includes imputation (even if no nulls left for demonstration) and scaling
numerical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')), # impute median in case of future nulls
    ('scaler', StandardScaler()) # scale numerical features
])

# create a column transformer to apply transformations to numerical columns
preprocessor = ColumnTransformer(
    transformers=[
        ('num', numerical_transformer, numerical_cols)
    ])

print('preprocessing pipeline for numerical features created')
print(f'columns to be processed: {numerical_cols}')

preprocessing pipeline for numerical features created
columns to be processed: ['Pclass', 'Age', 'SibSp', 'Parch', 'Fare']


In [17]:
# apply the preprocessor to the training data
x_train_processed = preprocessor.fit_transform(x_train)

# apply the fitted preprocessor to the test data
x_test_processed = preprocessor.transform(x_test)

print('training and test data processed through the pipeline')
print('first 5 rows of processed x_train:')
display(pd.DataFrame(x_train_processed, columns=numerical_cols).head())

training and test data processed through the pipeline
first 5 rows of processed x_train:


,Pclass,Age,SibSp,Parch,Fare
0,-1.614136,1.362465,-0.470722,-0.479342,0.224500
1,-0.400551,-0.488196,-0.470722,-0.479342,-0.531688
2,0.813034,0.252069,-0.470722,-0.479342,-0.779279
3,0.813034,-0.241441,0.379923,-0.479342,-0.782733
4,0.813034,-1.886473,2.931860,2.048742,0.359882
